# Native LLM/VLM evaluation on Kaggle — two GPUs

Upload this notebook to Kaggle. In **Settings → Accelerator**, select **GPU T4 x2**,
enable **Internet**, and attach your VQA dataset. Edit the next cell and **Run All**.
The notebook contains the evaluator and shared metrics; no repository clone is needed.

The default is **LLaVA-OneVision-1.5-8B-Instruct** on **SciVQA test with full figure context**. Change `MODEL` to
select any of the six supported checkpoints below. For GQA or PlantExpertVQA, change `DATASET` and
`DATA_ROOT`. SciVQA includes answer options, captions, figure types, and compound-figure guidance.
The default `SPLIT="test"` and `FULL_LIMIT=None` evaluate all 4,200 test questions.
Use `SPLIT="dev"` for developing model/prompt settings. A five-question smoke run precedes full evaluation. Each run writes
only `predictions.jsonl`, `report.json`, and `run_config.json`.

Weights use native Transformers and Accelerate's `device_map="balanced"` across
the two GPUs. T4 uses FP16 and SDPA. The dependency cell selects Transformers 4.57.1 for
OneVision 1.5 and Transformers 5.x for the other presets. OneVision loads its
custom Python code from the same pinned Hugging Face commit as its weights. Weights must fit within the GPU budgets;
image inputs and generation also need memory. The official Ministral FP8 checkpoint
is dequantized while loading. A smoke run checks the workflow on your hardware.

[Kaggle GPU setup](https://www.kaggle.com/docs/notebooks) ·
[Accelerate model placement](https://huggingface.co/docs/accelerate/main/concept_guides/big_model_inference)


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

MODEL = "lmms-lab/LLaVA-OneVision-1.5-8B-Instruct"
# MODEL = "Qwen/Qwen3-VL-8B-Instruct"
# MODEL = "OpenGVLab/InternVL3_5-8B-HF"
# MODEL = "mistralai/Ministral-3-8B-Instruct-2512"
# MODEL = "zai-org/GLM-4.6V-Flash"
# MODEL = "google/gemma-4-E4B-it"
# Or use a local folder: MODEL = "/kaggle/input/my-native-checkpoint/model"

DATASET = "scivqa"  # "scivqa", "plantexpert", or "gqa"
SPLIT = "test"  # SciVQA final evaluation; use "dev" explicitly for validation.
SCIVQA_CONTEXT = "full"  # "question-only" retains choices but excludes figure context.
DATA_ROOT = Path("/kaggle/input/scivqa-dataset/scivqa_dataset")
QUESTIONS = DATA_ROOT / ("val.csv" if SPLIT == "dev" else "test.csv")
IMAGES = DATA_ROOT / "images"
IMAGE_SPLIT = "val" if SPLIT == "dev" else "test"  # Fallback folder for flat image names.
HF_REVISION = "main"  # A saved run's pinned commit is reused when resuming main.
DTYPE = "auto"  # FP16 on T4; BF16 on Ampere or newer.
GPU_MEMORY_GIB = 12  # Weight budget on each GPU; leave room for images/KV cache.
ATTENTION_IMPLEMENTATION = "sdpa"
CONTEXT_SIZE = 8192
MAX_TOKENS = 64 if DATASET == "gqa" else 256
NO_BERTSCORE = False  # PlantExpert/SciVQA: True skips semantic metrics.
RUN_SMOKE = True
FULL_LIMIT = None  # None = all questions; a positive integer limits evaluation.
USE_KAGGLE_HF_TOKEN = False  # Optional for public models; required for gated models such as Gemma.

WORK = Path("/kaggle/working")
EVAL_ROOT = WORK / "ot-vqa-eval"
RESULTS_ROOT = WORK / "results"
CACHE_DIR = WORK / "hf-cache"
ENV = os.environ.copy()
ENV["CUDA_VISIBLE_DEVICES"] = "0,1"
ENV["TOKENIZERS_PARALLELISM"] = "false"
assert DATASET in {"gqa", "plantexpert", "scivqa"}
if USE_KAGGLE_HF_TOKEN:
    from kaggle_secrets import UserSecretsClient
    ENV["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("Model:", MODEL, "\nQuestions:", QUESTIONS, "\nImages:", IMAGES)

def subprocess_json(arguments):
    result = subprocess.run([sys.executable, *arguments], cwd=EVAL_ROOT, env=ENV,
                            text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError("Model setup failed:\n" + result.stderr + result.stdout)
    if result.stderr.strip():
        print(result.stderr, end="")
    return json.loads(result.stdout)


In [ ]:
BUNDLED_SOURCES = {'scripts/evaluate_llm.py': '"""Zero-shot VQA evaluation with native Hugging Face vision-language checkpoints.\n\nRun from the repository root: python3 -m scripts.evaluate_llm --help\nNo GGUF weights, local server, or model conversion is used.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport math\nimport re\nimport sys\nimport time\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom urllib.parse import urlparse\n\nfrom utils.metrics import (PAPER_METRICS, SCIVQA_METRICS, bertscore_defaults,\n                           build_bertscore_scorer, lexical_scores,\n                           normalize_llm_answer, scivqa_score_pairs, score_pairs)\nfrom utils import scivqa_prompt\n\nMODEL_PRESETS = {\n    "llava-onevision-1.5-8b": "lmms-lab/LLaVA-OneVision-1.5-8B-Instruct",\n    "qwen3-vl-8b": "Qwen/Qwen3-VL-8B-Instruct",\n    "internvl3.5-8b": "OpenGVLab/InternVL3_5-8B-HF",\n    "ministral-3-8b": "mistralai/Ministral-3-8B-Instruct-2512",\n    "glm-4.6v-flash": "zai-org/GLM-4.6V-Flash",\n    "gemma-4-e4b": "google/gemma-4-E4B-it",\n}\nMODEL_CLASSES = {\n    "llavaonevision1_5": "AutoModelForCausalLM",\n    "qwen3_vl": "Qwen3VLForConditionalGeneration",\n    "internvl": "InternVLForConditionalGeneration",\n    "mistral3": "Mistral3ForConditionalGeneration",\n    "glm4v": "Glm4vForConditionalGeneration",\n    "gemma4": "Gemma4ForConditionalGeneration",\n}\nONEVISION_MODEL_TYPE = "llavaonevision1_5"\nONEVISION_AUTO_MAP = {\n    "AutoConfig": "configuration_llavaonevision1_5.Llavaonevision1_5Config",\n    "AutoModelForCausalLM": "modeling_llavaonevision1_5.LLaVAOneVision1_5_ForConditionalGeneration",\n}\n\n\ndef normalize_model_config(config: dict) -> dict:\n    """Identify OneVision 1.5 even when its published config omits model_type."""\n    if not isinstance(config, dict):\n        raise ValueError("Model config.json must contain a JSON object.")\n    config = config.copy()\n    auto_map = config.get("auto_map") or {}\n    onevision = all(auto_map.get(key) == value for key, value in ONEVISION_AUTO_MAP.items())\n    if onevision and not config.get("model_type"):\n        config["model_type"] = ONEVISION_MODEL_TYPE\n    if config.get("model_type") == ONEVISION_MODEL_TYPE and not onevision:\n        raise ValueError("OneVision 1.5 requires the checkpoint\'s original AutoConfig/AutoModelForCausalLM mappings.")\n    if config.get("model_type") not in MODEL_CLASSES:\n        raise ValueError(f"Unsupported vision architecture: {config.get(\'model_type\')!r}; "\n                         f"architectures={config.get(\'architectures\', [])}. Use --list-models. "\n                         "InternVL3.5 requires OpenGVLab/InternVL3_5-8B-HF.")\n    return config\n\n\ndef runtime_requirements(config: dict) -> list[str]:\n    """OneVision\'s custom implementation needs the Transformers 4.x APIs."""\n    if config["model_type"] == ONEVISION_MODEL_TYPE:\n        return ["transformers==4.57.1", "huggingface-hub>=0.34,<1"]\n    return ["transformers>=5.5,<6", "huggingface-hub>=1,<2"]\n\nSYSTEM_PROMPT = (\n    "Answer the question using the image. Return only the short answer, "\n    "usually one word or a short phrase. For yes/no questions, return yes or no. "\n    "Do not include an explanation, a full sentence, or an \'Answer:\' prefix."\n)\nPLANTEXPERT_SYSTEM_PROMPT = (\n    "Answer the plant-science question using the image. Give a concise, complete "\n    "answer with the details requested by the question. For yes/no questions, "\n    "answer yes or no and explain only if requested. Do not include an \'Answer:\' "\n    "prefix, unrelated information, or discussion of your reasoning process."\n)\nMETADATA_FIELDS = ("question_type", "question_category", "crop", "disease")\nANSWER_NORMALIZATION = "whitespace_markdown_v1"\nNORMALIZATION_DESCRIPTION = (\n    "Predictions and references have actual/literal escaped newlines and tabs, "\n    "repeated whitespace, and Markdown emphasis, code fences, headings, and bullets "\n    "normalized before scoring. Meaningful punctuation and case are preserved."\n)\n\n\n@dataclass(frozen=True)\nclass Example:\n    question_id: str\n    image: str\n    question: str\n    answer: str | None\n    metadata: dict[str, str] = field(default_factory=dict)\n\n\ndef load_examples(path: Path, dataset: str = "gqa", context_mode: str = "full") -> list[Example]:\n    """Accept repository CSVs or official GQA questionId -> question JSON."""\n    examples = []\n    if path.suffix.lower() == ".csv":\n        with path.open(encoding="utf-8-sig", newline="") as handle:\n            reader = csv.DictReader(handle)\n            if not {"image", "question"}.issubset(reader.fieldnames or []):\n                raise ValueError("CSV must contain image and question columns; answer is optional.")\n            if dataset == "scivqa":\n                required = {"original_question", "answer_options", "qa_pair_type"}\n                if context_mode == "full":\n                    required.update({"caption", "figure_type", "compound", "figs_numb"})\n                missing = required - set(reader.fieldnames)\n                if missing:\n                    raise ValueError(f"SciVQA context columns missing: {sorted(missing)}. "\n                                     "Use the CSVs exported by utils.download_scivqa.")\n            for index, row in enumerate(reader):\n                question = row["question"]\n                fields = METADATA_FIELDS\n                if dataset == "scivqa":\n                    question = scivqa_prompt.build_prompt(question, row, context_mode)\n                    fields = (*scivqa_prompt.CONTEXT_FIELDS, *scivqa_prompt.PROVENANCE_FIELDS)\n                examples.append(Example(\n                    row.get("question_id") or row.get("anno_id") or str(index),\n                    row["image"], question, row.get("answer") or None,\n                    {key: row[key] for key in fields if key in row},\n                ))\n    elif path.suffix.lower() == ".json":\n        if dataset == "scivqa":\n            raise ValueError("SciVQA requires an exported context-preserving CSV, not GQA JSON.")\n        data = json.loads(path.read_text(encoding="utf-8"))\n        if not isinstance(data, dict):\n            raise ValueError("GQA JSON must map question IDs to question objects.")\n        for question_id, row in data.items():\n            examples.append(Example(\n                str(question_id), f"{row[\'imageId\']}.jpg", row["question"],\n                row.get("answer") or None,\n            ))\n    else:\n        raise ValueError("Questions must be a .csv or .json file.")\n    if not examples:\n        raise ValueError("The question file is empty.")\n    ids = set()\n    for example in examples:\n        if example.question_id in ids:\n            raise ValueError(f"Duplicate question ID: {example.question_id}")\n        ids.add(example.question_id)\n        if not example.image or not example.question.strip():\n            raise ValueError(f"Missing image or question for {example.question_id}")\n    return examples\n\n\ndef image_path(root: Path, example: Example, split: str) -> Path:\n    """Resolve image.jpg, split/image.jpg, and repository split-folder layouts."""\n    direct = root / example.image\n    if direct.is_file():\n        return direct\n    nested = root / split / example.image\n    if nested.is_file():\n        return nested\n    raise FileNotFoundError(f"Image not found: {direct} (also checked {nested})")\n\n\ndef normalize(text: str) -> str:\n    return normalize_llm_answer(text).casefold()\n\n\ndef make_record(example: Example, result: dict | None = None, error: str = "",\n                dataset: str = "gqa") -> dict:\n    record = {\n        "question_id": example.question_id, "image": example.image,\n        "question": example.question, "reference": example.answer,\n        "prediction": "", "raw_response": "", "error": error,\n    }\n    record.update(example.metadata)\n    record.update(result or {})\n    record["prediction"] = normalize_llm_answer(record["prediction"])\n    labeled = example.answer is not None\n    reference = normalize_llm_answer(example.answer) if labeled else None\n    if dataset == "scivqa":\n        metrics = SCIVQA_METRICS[:6]\n        scores = {metric: None for metric in metrics}\n        if labeled:\n            scores = ({metric: 0.0 for metric in metrics} if error else\n                      scivqa_score_pairs([reference], [record["prediction"]], None,\n                                        compute_bertscore=False)[0])\n        record.update(scores)\n        return record\n    if dataset == "plantexpert":\n        if not labeled:\n            scores = {metric: None for metric in PAPER_METRICS[:-1]}\n        elif error:\n            scores = {metric: 0.0 for metric in PAPER_METRICS[:-1]}\n        else:\n            scores = lexical_scores(reference, record["prediction"])\n        record.update(scores)\n        return record\n    record["correct"] = (\n        bool(not error and record["prediction"] == reference) if labeled else None\n    )\n    record["normalized_correct"] = (\n        bool(not error and normalize(record["prediction"]) == normalize(example.answer))\n        if labeled else None\n    )\n    return record\n\n\ndef save_outputs(output: Path, records: dict[str, dict], config: dict,\n                 selected_count: int, bert_scorer=None) -> dict:\n    rows = list(records.values())\n    labeled = [row for row in rows if row["reference"] is not None]\n    successful = [row for row in rows if not row["error"]]\n    report = {\n        "config": config,\n        "examples_selected": selected_count,\n        "examples_attempted": len(rows),\n        "examples_succeeded": len(successful),\n        "errors": len(rows) - len(successful),\n        "labeled_examples_attempted": len(labeled),\n        "complete": len(successful) == selected_count,\n        "truncated_responses": sum(row.get("finish_reason") == "length" for row in successful),\n        "mean_latency_seconds": (\n            sum(row["latency_seconds"] for row in successful) / len(successful)\n            if successful else None\n        ),\n    }\n    plantexpert = config.get("dataset") == "plantexpert"\n    scivqa = config.get("dataset") == "scivqa"\n    if plantexpert or scivqa:\n        lexical_metrics = SCIVQA_METRICS[:6] if scivqa else PAPER_METRICS[:-1]\n        semantic_metrics = SCIVQA_METRICS[6:] if scivqa else ("bertscore_f1",)\n        report["generated_metrics"] = {\n            metric: sum(row[metric] for row in labeled) / len(labeled) if labeled else None\n            for metric in lexical_metrics\n        }\n        report["generated_metrics"].update({metric: None for metric in semantic_metrics})\n        report["bertscore"] = config["bertscore"].copy()\n        report["scoring"] = {\n            "metrics": ("Repository SciVQA ROUGE-1/L and BERTScore precision, recall and F1; macro means."\n                        if scivqa else "Repository PlantExpert metrics; scores are fractions, rescaled BERTScore can be negative."),\n            "errors": "Failed labeled requests receive zero for every metric; unattempted questions are excluded.",\n        }\n    else:\n        report.update({\n            "accuracy": sum(row["correct"] for row in labeled) / len(labeled) if labeled else None,\n            "normalized_accuracy": (\n                sum(row["normalized_correct"] for row in labeled) / len(labeled) if labeled else None\n            ),\n            "scoring": {\n                "accuracy": "Exact match after answer formatting normalization; one point per correct answer.",\n                "normalized_accuracy": "Exact match after answer formatting normalization, also ignoring case.",\n                "errors": "Failed labeled requests count as incorrect; unattempted questions are excluded.",\n            },\n        })\n    report["scoring"]["answer_normalization"] = NORMALIZATION_DESCRIPTION\n    # Save a report before semantic scoring, so scoring failures preserve inference results.\n    (output / "report.json").write_text(json.dumps(report, indent=2) + "\\n", encoding="utf-8")\n    if plantexpert or scivqa:\n        for row in rows:\n            failed_label = row["error"] and row["reference"] is not None\n            for metric in semantic_metrics:\n                row[metric] = 0.0 if bert_scorer is not None and failed_label else None\n        if bert_scorer is not None:\n            scored = [row for row in labeled if not row["error"]]\n            if scored:\n                scorer = scivqa_score_pairs if scivqa else score_pairs\n                scores = scorer([normalize_llm_answer(row["reference"]) for row in scored],\n                                [row["prediction"] for row in scored], bert_scorer)\n                for row, metrics in zip(scored, scores):\n                    row.update(metrics)\n            for metric in semantic_metrics:\n                report["generated_metrics"][metric] = (\n                    sum(row[metric] for row in labeled) / len(labeled) if labeled else None\n                )\n            report["bertscore"]["model_hash"] = bert_scorer.hash\n        if scivqa:\n            report["by_qa_pair_type"] = {}\n            for kind in sorted({row.get("qa_pair_type", "unknown") for row in labeled}):\n                group = [row for row in labeled if row.get("qa_pair_type", "unknown") == kind]\n                report["by_qa_pair_type"][kind] = {\n                    "count": len(group),\n                    "generated_metrics": {\n                        metric: (sum(row[metric] for row in group) / len(group)\n                                 if all(row.get(metric) is not None for row in group) else None)\n                        for metric in SCIVQA_METRICS\n                    },\n                }\n        (output / "report.json").write_text(json.dumps(report, indent=2) + "\\n", encoding="utf-8")\n        return report\n    return report\n\n\ndef evaluate(args: argparse.Namespace, client: TransformersClient) -> dict:\n    examples = load_examples(args.questions, args.dataset, args.scivqa_context)\n    if args.limit is not None:\n        examples = examples[:args.limit]\n    # Validate the selected files before starting a potentially long inference run.\n    paths = {example.question_id: image_path(args.images, example, args.image_split or args.questions.stem)\n             for example in examples}\n    plantexpert = args.dataset == "plantexpert"\n    scivqa = args.dataset == "scivqa"\n    system_prompt = (scivqa_prompt.SYSTEM_PROMPT if scivqa else\n                     PLANTEXPERT_SYSTEM_PROMPT if plantexpert else SYSTEM_PROMPT)\n    config = {\n        "questions": str(args.questions.resolve()),\n        "questions_sha256": hashlib.sha256(args.questions.read_bytes()).hexdigest(),\n        "images": str(args.images.resolve()),\n        "image_split": args.image_split or args.questions.stem,\n        "model": args.model, "base_url": client.base_url,\n        "limit": args.limit, "seed": args.seed,\n        "temperature": 0, "max_tokens": args.max_tokens,\n        "system_prompt": system_prompt, "protocol": "zero-shot",\n        "answer_normalization": ANSWER_NORMALIZATION,\n    }\n    if getattr(args, "backend_config", None) is not None:\n        config["backend"] = args.backend_config\n    if scivqa:\n        config["context"] = {\n            "mode": args.scivqa_context, "prompt_version": scivqa_prompt.PROMPT_VERSION,\n            "fields": (list(scivqa_prompt.CONTEXT_FIELDS) if args.scivqa_context == "full"\n                       else ["original_question", "answer_options", "qa_pair_type"]),\n            "prompts_sha256": hashlib.sha256(json.dumps(\n                [(example.question_id, example.question) for example in examples],\n                ensure_ascii=False).encode("utf-8")).hexdigest(),\n            "qa_pair_type_usage": "binary/non-binary answer format only; never answerability",\n            "provenance_fields_excluded_from_prompt": list(scivqa_prompt.PROVENANCE_FIELDS),\n        }\n    if plantexpert or scivqa:\n        config.update({"dataset": args.dataset, "bertscore": {\n            "enabled": not args.no_bertscore, "model": args.bertscore_model,\n            "device": args.bertscore_device, "batch_size": args.bertscore_batch_size,\n            "rescale_with_baseline": args.bertscore_rescale,\n        }})\n    output = args.output_dir\n    output.mkdir(parents=True, exist_ok=True)\n    manifest = output / "run_config.json"\n    journal = output / "predictions.jsonl"\n    records = {}\n    if args.resume:\n        saved_config = json.loads(manifest.read_text(encoding="utf-8")) if manifest.is_file() else None\n        manifest_config = saved_config.copy() if saved_config is not None else None\n        # Older journals retain raw predictions; rescore them with current cleanup.\n        if saved_config is not None:\n            saved_config.setdefault("answer_normalization", ANSWER_NORMALIZATION)\n        if saved_config != config:\n            raise ValueError("Resume configuration differs or is missing. Use the original flags or a new output-dir.")\n        if journal.is_file():\n            examples_by_id = {example.question_id: example for example in examples}\n            for line in journal.read_text(encoding="utf-8").splitlines():\n                row = json.loads(line)\n                records[row["question_id"]] = make_record(\n                    examples_by_id[row["question_id"]], row, error=row["error"], dataset=args.dataset,\n                )\n    else:\n        if any(output.iterdir()):\n            raise ValueError(f"Output directory is not empty: {output}. Use --resume or a new output-dir.")\n    bert_scorer = None\n    if (plantexpert or scivqa) and not args.no_bertscore and any(example.answer is not None for example in examples):\n        print("Loading BERTScore before inference...", flush=True)\n        bert_scorer = build_bertscore_scorer(\n            model_type=args.bertscore_model, device=args.bertscore_device,\n            batch_size=args.bertscore_batch_size, rescale_with_baseline=args.bertscore_rescale,\n        )\n    if not args.resume:\n        manifest.write_text(json.dumps(config, indent=2) + "\\n", encoding="utf-8")\n    elif "answer_normalization" not in manifest_config:\n        manifest_config["answer_normalization"] = ANSWER_NORMALIZATION\n        manifest.write_text(json.dumps(manifest_config, indent=2) + "\\n", encoding="utf-8")\n    print(f"Model: {args.model}; selected questions: {len(examples)}; zero-shot", flush=True)\n    try:\n        with journal.open("a", encoding="utf-8") as handle:\n            for index, example in enumerate(examples, 1):\n                previous = records.get(example.question_id)\n                if previous is not None and not previous["error"]:\n                    continue\n                failure = None\n                try:\n                    prompt_args = {"system_prompt": system_prompt} if plantexpert or scivqa else {}\n                    result = client.answer(args.model, example, paths[example.question_id],\n                                           args.max_tokens, args.seed, **prompt_args)\n                    record = make_record(example, result, dataset=args.dataset)\n                except (RuntimeError, OSError, ValueError, KeyError, IndexError, TypeError) as error:\n                    failure = error\n                    record = make_record(example, error=str(error), dataset=args.dataset)\n                handle.write(json.dumps(record, ensure_ascii=False) + "\\n")\n                handle.flush()\n                records[example.question_id] = record\n                print(f"[{index}/{len(examples)}] {example.question_id}: "\n                      f"{record[\'prediction\']!r} | reference={example.answer!r}"\n                      + (f" | ERROR: {failure}" if failure else ""), flush=True)\n                if failure and not args.continue_on_error:\n                    raise RuntimeError("Stopped after an error; predictions are saved. Fix it and use --resume.") from failure\n    finally:\n        report = save_outputs(output, records, config, len(examples), bert_scorer)\n    return report\n\n\n\ndef inspect_model(model: str, revision: str = "main", cache_dir: Path | None = None) -> tuple[dict, dict]:\n    """Read config/metadata first and pin remote weights and processor together."""\n    model = MODEL_PRESETS.get(model.lower(), model)\n    local = Path(model).expanduser()\n    if local.exists():\n        if not local.is_dir():\n            raise ValueError("--model must be a native checkpoint folder or Hugging Face repository, not a GGUF file.")\n        names = [path.relative_to(local).as_posix() for path in local.rglob("*") if path.is_file()]\n        source = {"directory": str(local.resolve()), "names": names}\n        config_path = local / "config.json"\n    else:\n        if local.is_absolute() or model.startswith((".", "~")):\n            raise FileNotFoundError(f"Local model folder does not exist: {local}")\n        if model.startswith("https://huggingface.co/"):\n            model = urlparse(model).path.strip("/")\n        if not re.fullmatch(r"[\\w.-]+/[\\w.-]+", model) or "gguf" in model.lower():\n            raise ValueError("Use a native Hugging Face owner/model repository; GGUF and quantization suffixes are unsupported.")\n        from huggingface_hub import HfApi, hf_hub_download\n        api = HfApi()\n        commit = api.model_info(model, revision=revision).sha\n        if not commit:\n            raise ValueError("Cannot resolve the model revision to a commit.")\n        names = api.list_repo_files(model, revision=commit)\n        source = {"repo": model, "revision": commit, "names": names}\n        # Reject GGUF-only repositories before downloading anything.\n        validate_model_files(names)\n        config_path = Path(hf_hub_download(repo_id=model, filename="config.json", revision=commit,\n                                          cache_dir=str(cache_dir) if cache_dir else None))\n    validate_model_files(names)\n    config = normalize_model_config(json.loads(config_path.read_text(encoding="utf-8")))\n    return source, config\n\n\ndef validate_model_files(names: list[str]):\n    if "config.json" not in names or not any(\n        name in names for name in ("model.safetensors", "model.safetensors.index.json",\n                                   "pytorch_model.bin", "pytorch_model.bin.index.json")\n    ):\n        raise ValueError("Expected config.json and native Transformers weights (Safetensors or pytorch_model.bin). GGUF is unsupported.")\n\n\ndef resolve_model(source: dict, cache_dir: Path | None) -> tuple[Path, dict]:\n    """Download one Transformers weight set, excluding GGUF and duplicate Mistral consolidated weights."""\n    names = source["names"]\n    safe = "model.safetensors" in names or "model.safetensors.index.json" in names\n    weight_suffix = ".safetensors" if safe else ".bin"\n    assets = [name for name in names if name.endswith((".json", ".model", ".txt", ".jinja", ".tiktoken", ".py"))\n              or name.startswith("model" if safe else "pytorch_model") and name.endswith(weight_suffix)]\n    if "directory" in source:\n        directory = Path(source["directory"])\n        checksums = {}\n        for name in sorted(assets):\n            digest = hashlib.sha256()\n            with (directory / name).open("rb") as handle:\n                for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n                    digest.update(chunk)\n            checksums[name] = digest.hexdigest()\n        identity = {"directory": str(directory), "files_sha256": checksums}\n    else:\n        from huggingface_hub import snapshot_download\n        directory = Path(snapshot_download(repo_id=source["repo"], revision=source["revision"],\n                                          cache_dir=str(cache_dir) if cache_dir else None,\n                                          allow_patterns=assets))\n        identity = {"repo": source["repo"], "revision": source["revision"]}\n    return directory, identity\n\n\ndef validate_runtime(config: dict, dtype: str):\n    try:\n        import torch\n        import accelerate\n        import transformers\n    except ImportError as error:\n        raise RuntimeError("Install evaluation dependencies: " + " ".join(runtime_requirements(config))\n                           + " accelerate pillow safetensors") from error\n    from packaging.specifiers import SpecifierSet\n    requirement = runtime_requirements(config)[0]\n    if transformers.__version__ not in SpecifierSet(requirement.removeprefix("transformers")):\n        raise RuntimeError(f"{config[\'model_type\']} needs {requirement}; installed {transformers.__version__}. "\n                           "Run: pip install -U " + " ".join(f"\'{item}\'" for item in runtime_requirements(config)))\n    count = torch.cuda.device_count()\n    if not count:\n        raise RuntimeError("An NVIDIA CUDA GPU is required. On Kaggle select GPU T4 x2.")\n    bf16 = all(torch.cuda.get_device_capability(i)[0] >= 8 for i in range(count))\n    if dtype == "auto":\n        dtype = "bfloat16" if bf16 else "float16"\n    if dtype == "bfloat16" and not bf16:\n        raise ValueError("bfloat16 requires Ampere or newer GPUs; select float16 for T4.")\n    model_class = MODEL_CLASSES[config["model_type"]]\n    loader = getattr(transformers, model_class, None)\n    if loader is None:\n        raise RuntimeError(f"Installed Transformers {transformers.__version__} lacks {model_class}; upgrade transformers.")\n    loading = {}\n    quantization = config.get("quantization_config") or {}\n    if quantization:\n        if quantization.get("quant_method") != "fp8":\n            raise ValueError("Use the original native checkpoint; only its FP8-to-16-bit loading is supported.")\n        # Official Ministral weights are FP8. Dequantize during loading for T4.\n        loading["quantization_config"] = transformers.FineGrainedFP8Config(dequantize=True)\n    return torch, accelerate, transformers, count, dtype, loader, loading\n\n\nclass TransformersClient:\n    base_url = "transformers://local"\n\n    def __init__(self, directory: Path, alias: str, config: dict, dtype: str = "auto",\n                 gpu_memory_gib: float = 12, context_size: int = 8192, attention: str = "sdpa"):\n        torch, accelerate, transformers, count, dtype, loader, loading = validate_runtime(config, dtype)\n        model_class = MODEL_CLASSES[config["model_type"]]\n        quantization = config.get("quantization_config") or {}\n        self.torch = torch\n        self.alias = alias\n        self.context_size = context_size\n        self.onevision = config["model_type"] == ONEVISION_MODEL_TYPE\n        # Only this explicitly supported custom architecture executes checkpoint code.\n        # resolve_model downloads its Python files at the same pinned commit as the weights.\n        self.processor = transformers.AutoProcessor.from_pretrained(\n            str(directory), trust_remote_code=self.onevision, local_files_only=True,\n            **({"use_fast": False} if self.onevision else {}),\n        )\n        self.model = loader.from_pretrained(\n            str(directory), dtype=getattr(torch, dtype), device_map="balanced",\n            max_memory={**{i: int(gpu_memory_gib * 2**30) for i in range(count)}, "cpu": 0},\n            attn_implementation=attention, trust_remote_code=self.onevision, local_files_only=True,\n            **loading,\n        ).eval()\n        placement = getattr(self.model, "hf_device_map", {})\n        allowed = {str(i) for i in range(count)} | {f"cuda:{i}" for i in range(count)}\n        if not placement or any(str(device) not in allowed for device in placement.values()):\n            raise RuntimeError("Weights do not fit in the GPU budget. Increase --gpu-memory-gib or use GPUs with more memory.")\n        self.backend_config = {\n            "engine": "transformers", "version": transformers.__version__,\n            "torch_version": torch.__version__, "accelerate_version": accelerate.__version__,\n            "model_class": model_class, "dtype": dtype, "attention_implementation": attention,\n            "device_map": {name: str(device) for name, device in placement.items()},\n            "gpu_memory_gib": gpu_memory_gib, "context_size": context_size,\n            "fp8_dequantized": bool(quantization), "enable_thinking": False,\n        }\n        if self.onevision:\n            self.backend_config.update({"trust_remote_code": True,\n                                        "model_class": type(self.model).__name__})\n\n    def answer(self, model, example, path, max_tokens, seed, system_prompt=SYSTEM_PROMPT) -> dict:\n        if model != self.alias:\n            raise ValueError(f"Loaded model is {self.alias}, not {model}.")\n        start = time.perf_counter()\n        messages = [\n            {"role": "system", "content": [{"type": "text", "text": system_prompt}]},\n            {"role": "user", "content": [\n                {"type": "image", "url": str(path.resolve())},\n                {"type": "text", "text": example.question},\n            ]},\n        ]\n        if self.onevision:\n            # The official checkpoint uses Qwen2.5-VL\'s processor with temporal_patch_size=1.\n            # Feed the PIL image directly so its own resize/normalization settings are used.\n            from PIL import Image\n            text = self.processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)\n            with Image.open(path) as image:\n                inputs = self.processor(text=[text], images=[image.convert("RGB")],\n                                        padding=True, return_tensors="pt")\n        else:\n            inputs = self.processor.apply_chat_template(\n                messages, add_generation_prompt=True, tokenize=True,\n                return_dict=True, return_tensors="pt", enable_thinking=False,\n            )\n        prompt_tokens = inputs["input_ids"].shape[-1]\n        if prompt_tokens + max_tokens > self.context_size:\n            raise ValueError(f"Image/question uses {prompt_tokens} tokens plus {max_tokens} output tokens, "\n                             f"exceeding --context-size {self.context_size}.")\n        # Preserve integer token IDs, masks, grids, and other processor metadata.\n        inputs = {key: (value.to(device=self.model.device, dtype=self.model.dtype)\n                        if value.is_floating_point() else value.to(self.model.device))\n                  if self.torch.is_tensor(value) else value for key, value in inputs.items()}\n        self.torch.manual_seed(seed)\n        with self.torch.inference_mode():\n            outputs = self.model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False,\n                                          num_beams=1, use_cache=True, return_dict_in_generate=False)\n        generated = outputs[0, prompt_tokens:]\n        raw = self.processor.decode(generated, skip_special_tokens=True)\n        if not isinstance(raw, str) or not raw.strip():\n            raise RuntimeError("The model returned no answer text. Check max-tokens and model settings.")\n        eos = self.model.generation_config.eos_token_id\n        eos = [eos] if isinstance(eos, int) else eos or []\n        ended = len(generated) > 0 and generated[-1].item() in eos\n        return {\n            "raw_response": raw, "prediction": normalize_llm_answer(raw),\n            "finish_reason": "length" if len(generated) >= max_tokens and not ended else "stop",\n            "latency_seconds": time.perf_counter() - start,\n            "usage": {"prompt_tokens": prompt_tokens, "completion_tokens": len(generated),\n                      "total_tokens": prompt_tokens + len(generated)},\n        }\n\n\ndef parse_args(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--model", "--model-dir", default=MODEL_PRESETS["qwen3-vl-8b"],\n                        help="Preset alias, native HF repository ID/URL, or local checkpoint folder")\n    parser.add_argument("--list-models", action="store_true", help="Show native model presets without downloading")\n    parser.add_argument("--hf-revision", default="main", help="HF revision; resolved to a commit for reproducibility")\n    parser.add_argument("--cache-dir", type=Path, help="HF cache directory (default: standard HF cache)")\n    parser.add_argument("--dtype", choices=("auto", "float16", "bfloat16", "float32"), default="auto")\n    parser.add_argument("--gpu-memory-gib", type=float, default=12, help="Weight budget per visible GPU; reserve room for generation")\n    parser.add_argument("--attention-implementation", choices=("sdpa", "eager"), default="sdpa")\n    parser.add_argument("--context-size", type=int, default=8192, help="Maximum prompt plus output tokens")\n    parser.add_argument("--dataset", choices=("gqa", "plantexpert", "scivqa"), default="gqa")\n    parser.add_argument("--split", choices=("dev", "test"), default="test",\n                        help="Default: test; dev reads val.csv")\n    parser.add_argument("--scivqa-context", choices=("full", "question-only"), default="full",\n                        help="SciVQA: include figure context by default; question-only retains options")\n    parser.add_argument("--preview-prompts", type=int, metavar="N",\n                        help="Print N model prompts and exit without loading models or scorers")\n    parser.add_argument("--questions", type=Path, help="Default: data/<dataset>_dataset/<split>.csv")\n    parser.add_argument("--images", type=Path, help="Default: data/<dataset>_dataset/images")\n    parser.add_argument("--image-split", help="Optional split subfolder for flat image names")\n    parser.add_argument("--output-dir", type=Path, help="Default: results/<model>_<dataset>_zero_shot")\n    parser.add_argument("--limit", type=int, help="First N questions (default: all)")\n    parser.add_argument("--max-tokens", type=int, help="Default: 64 for GQA, 256 for PlantExpert/SciVQA")\n    parser.add_argument("--no-bertscore", action="store_true", help="PlantExpert/SciVQA: skip semantic metrics")\n    parser.add_argument("--bertscore-model", help="Default: roberta-large for SciVQA, bert-base-uncased otherwise")\n    parser.add_argument("--bertscore-device", default="cpu")\n    parser.add_argument("--bertscore-batch-size", type=int, default=16)\n    rescale = parser.add_mutually_exclusive_group()\n    rescale.add_argument("--bertscore-rescale", dest="bertscore_rescale", action="store_true")\n    rescale.add_argument("--no-bertscore-rescale", dest="bertscore_rescale", action="store_false")\n    parser.set_defaults(bertscore_rescale=None)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--resume", action="store_true", help="Reuse saved successes and retry failed questions")\n    parser.add_argument("--continue-on-error", action="store_true", help="Record failures as incorrect and continue")\n    args = parser.parse_args(argv)\n    args.model = MODEL_PRESETS.get(args.model.lower(), args.model)\n    split_file = "val" if args.split == "dev" else "test"\n    args.questions = args.questions or Path(f"data/{args.dataset}_dataset/{split_file}.csv")\n    args.images = args.images or Path(f"data/{args.dataset}_dataset/images")\n    model_name = Path(args.model.rstrip("/")).name\n    slug = re.sub(r"[^a-z0-9]+", "_", model_name.lower()).strip("_") or "model"\n    suffix = f"_{args.split}_{args.scivqa_context}" if args.dataset == "scivqa" else ""\n    args.output_dir = args.output_dir or Path(f"results/{slug}_{args.dataset}{suffix}_zero_shot")\n    bert_model, bert_rescale = bertscore_defaults(args.dataset)\n    args.bertscore_model = args.bertscore_model or bert_model\n    if args.bertscore_rescale is None:\n        args.bertscore_rescale = bert_rescale\n    if args.max_tokens is None:\n        args.max_tokens = 64 if args.dataset == "gqa" else 256\n    if args.limit is not None and args.limit <= 0:\n        parser.error("--limit must be positive")\n    if args.preview_prompts is not None and args.preview_prompts <= 0:\n        parser.error("--preview-prompts must be positive")\n    if min(args.max_tokens, args.context_size, args.bertscore_batch_size) <= 0:\n        parser.error("max-tokens, context-size, and bertscore-batch-size must be positive")\n    if not math.isfinite(args.gpu_memory_gib) or args.gpu_memory_gib <= 0:\n        parser.error("gpu-memory-gib must be a finite positive number")\n    if args.max_tokens >= args.context_size:\n        parser.error("max-tokens must leave room for the prompt in context-size")\n    return args\n\n\ndef main(argv=None) -> int:\n    args = parse_args(argv)\n    if args.list_models:\n        for alias, repo in MODEL_PRESETS.items():\n            print(f"{alias}: {repo}")\n        return 0\n    try:\n        # Stop on bad data or an occupied output directory before model downloads.\n        examples = load_examples(args.questions, args.dataset, args.scivqa_context)[:args.limit]\n        for example in examples:\n            image_path(args.images, example, args.image_split or args.questions.stem)\n        if args.preview_prompts:\n            system_prompt = (scivqa_prompt.SYSTEM_PROMPT if args.dataset == "scivqa" else\n                             PLANTEXPERT_SYSTEM_PROMPT if args.dataset == "plantexpert" else SYSTEM_PROMPT)\n            print(json.dumps([{"question_id": example.question_id, "image": example.image,\n                               "system_prompt": system_prompt, "user_prompt": example.question}\n                              for example in examples[:args.preview_prompts]], ensure_ascii=False, indent=2))\n            return 0\n        if args.dataset == "scivqa":\n            # Check the lightweight lexical dependency before downloading weights.\n            scivqa_score_pairs([], [], None, compute_bertscore=False)\n        if not args.resume and args.output_dir.exists() and any(args.output_dir.iterdir()):\n            raise ValueError(f"Output directory is not empty: {args.output_dir}. Use --resume or a new output-dir.")\n        source, config = inspect_model(args.model, args.hf_revision, args.cache_dir)\n        validate_runtime(config, args.dtype)\n        directory, identity = resolve_model(source, args.cache_dir)\n        client = TransformersClient(directory, args.model, config, args.dtype, args.gpu_memory_gib,\n                                    args.context_size, args.attention_implementation)\n        args.backend_config = {**client.backend_config, "model_source": identity}\n        report = evaluate(args, client)\n        if args.dataset in {"plantexpert", "scivqa"}:\n            print(f"{args.dataset} metrics: " + ", ".join(\n                f"{key}={value:.4f}" if value is not None else f"{key}=not computed"\n                for key, value in report["generated_metrics"].items()))\n        elif report["accuracy"] is not None:\n            print(f"Accuracy: {report[\'accuracy\']:.2%}; normalized: {report[\'normalized_accuracy\']:.2%}")\n        else:\n            print("No reference answers supplied; predictions saved without scoring.")\n        print(f"Results: {args.output_dir / \'report.json\'}")\n        return 0 if report["complete"] else 1\n    except KeyboardInterrupt:\n        print("Interrupted. Rerun with identical settings and --resume.", file=sys.stderr)\n        return 130\n    except (RuntimeError, OSError, ValueError, KeyError, TypeError, ImportError) as error:\n        print(f"Error: {error}", file=sys.stderr)\n        return 1\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n', 'utils/metrics.py': '"""Dataset-specific generated-answer metrics for GQA, PlantExpertVQA and SciVQA.\n\nScores are fractions (0 to 1 for lexical metrics), not percentages. BERTScore\ncan be negative when baseline rescaling is enabled. The paper gives formulas\nbut not its tokenizer or BERTScore encoder; those choices are explicit here.\n"""\n\nfrom __future__ import annotations\n\nimport math\nimport re\nimport string\nfrom collections import Counter\nfrom statistics import fmean\n\n\nPAPER_METRICS = ("em", "token_f1", "bleu_1", "bleu_2", "rouge_l", "bertscore_f1")\nGQA_METRICS = ("accuracy",)\nSCIVQA_METRICS = (\n    "rouge_1_f1", "rouge_1_precision", "rouge_1_recall",\n    "rouge_l_f1", "rouge_l_precision", "rouge_l_recall",\n    "bertscore_f1", "bertscore_precision", "bertscore_recall",\n)\nALL_METRICS = tuple(dict.fromkeys((*GQA_METRICS, *PAPER_METRICS, *SCIVQA_METRICS)))\n_ARTICLE = re.compile(r"\\b(a|an|the)\\b", flags=re.IGNORECASE)\n_WORD_PUNCT = re.compile(r"\\w+|[^\\w\\s]", flags=re.UNICODE)\n\n\ndef resolve_dataset(dataset="auto", csv_path=None):\n    """Use an explicit dataset or detect GQA/SciVQA in a CSV path."""\n    dataset = dataset.lower()\n    if dataset not in {"auto", "gqa", "plantexpert", "scivqa"}:\n        raise ValueError(f"Unknown dataset: {dataset}")\n    if dataset != "auto":\n        return dataset\n    for name in ("scivqa", "gqa"):\n        if csv_path is not None and re.search(\n                rf"(?:^|[^a-z0-9]){name}(?:$|[^a-z0-9])", str(csv_path).lower()):\n            return name\n    return "plantexpert"\n\n\ndef metrics_for_dataset(dataset):\n    return {"gqa": GQA_METRICS, "plantexpert": PAPER_METRICS,\n            "scivqa": SCIVQA_METRICS}[resolve_dataset(dataset)]\n\n\ndef bertscore_defaults(dataset):\n    """Explicit project defaults; the SciVQA paper does not specify its encoder."""\n    return ("roberta-large", False) if resolve_dataset(dataset) == "scivqa" else (\n        "bert-base-uncased", True)\n\n\ndef normalize_text(text):\n    """Light normalization retained for dataset diagnostics."""\n    return " ".join(str(text).casefold().strip().split())\n\n\ndef normalize_llm_answer(text: str) -> str:\n    """Remove LLM whitespace/Markdown formatting while retaining answer content.\n\n    Handle actual and literal escaped newlines/tabs without decoding arbitrary\n    escapes or removing meaningful punctuation, numbers, Unicode, or case.\n    """\n    text = re.sub(r"\\\\(?:r\\\\n|[nrt])", "\\n", text)\n    text = re.sub(r"(?m)^[ \\t]*(?:`{3,}|~{3,})[ \\t]*(?:[\\w+-]+)?[ \\t]*$", "", text)\n    text = re.sub(r"(?m)^[ \\t]{0,3}(?:#{1,6}[ \\t]+|>[ \\t]?|[-+*][ \\t]+)", "", text)\n    for marker in ("**", "__", "~~", "```", "``", "`", "*", "_"):\n        escaped = re.escape(marker)\n        boundary = r"(?<!\\w)" if marker in {"*", "_", "__"} else ""\n        ending = r"(?!\\w)" if marker in {"*", "_", "__"} else ""\n        text = re.sub(boundary + escaped + r"(.+?)" + escaped + ending,\n                      r"\\1", text, flags=re.DOTALL)\n    return " ".join(text.split())\n\n\ndef gqa_accuracy(reference, hypothesis):\n    """Binary GQA exact match against one reference, with surrounding whitespace trimmed.\n\n    A matching generated answer receives 1 and a wrong answer receives 0.\n    Case, punctuation, and internal whitespace remain significant; answers are\n    not extracted from prose, mapped to synonyms, or scored by annotator votes.\n    """\n    if not isinstance(reference, str) or not isinstance(hypothesis, str):\n        raise TypeError("GQA accuracy requires one reference string and one prediction string")\n    return float(reference.strip() == hypothesis.strip())\n\n\ndef gqa_score_pairs(references, hypotheses):\n    """Per-example binary accuracy for aligned GQA references and predictions."""\n    if len(references) != len(hypotheses):\n        raise ValueError("References and hypotheses must have equal lengths")\n    return [{"accuracy": gqa_accuracy(ref, hyp)}\n            for ref, hyp in zip(references, hypotheses)]\n\n\ndef _as_text(value):\n    return value if isinstance(value, str) else " ".join(value)\n\n\ndef _squad_tokens(text):\n    text = _as_text(text).lower()\n    text = "".join(character for character in text if character not in string.punctuation)\n    text = _ARTICLE.sub(" ", text)\n    return text.split()\n\n\ndef _word_tokens(text):\n    return _WORD_PUNCT.findall(_as_text(text).casefold())\n\n\ndef _token_f1(reference, hypothesis):\n    ref, hyp = _squad_tokens(reference), _squad_tokens(hypothesis)\n    if not ref or not hyp:\n        return float(ref == hyp)\n    common = sum((Counter(ref) & Counter(hyp)).values())\n    return 2 * common / (len(ref) + len(hyp))\n\n\ndef _bleu(reference, hypothesis, order):\n    ref, hyp = _word_tokens(reference), _word_tokens(hypothesis)\n    if not ref or not hyp:\n        return float(ref == hyp)\n    precisions = []\n    for n in range(1, order + 1):\n        ref_ngrams = Counter(tuple(ref[i:i + n]) for i in range(len(ref) - n + 1))\n        hyp_ngrams = Counter(tuple(hyp[i:i + n]) for i in range(len(hyp) - n + 1))\n        if not hyp_ngrams:\n            return 0.0\n        matches = sum((ref_ngrams & hyp_ngrams).values())\n        if matches == 0:\n            return 0.0\n        precisions.append(matches / sum(hyp_ngrams.values()))\n    brevity_penalty = min(1.0, math.exp(1 - len(ref) / len(hyp)))\n    return brevity_penalty * math.exp(sum(math.log(p) for p in precisions) / order)\n\n\ndef _rouge_l(reference, hypothesis):\n    ref, hyp = _word_tokens(reference), _word_tokens(hypothesis)\n    if not ref or not hyp:\n        return float(ref == hyp)\n    previous = [0] * (len(hyp) + 1)\n    for ref_token in ref:\n        current = [0]\n        for column, hyp_token in enumerate(hyp, 1):\n            current.append(previous[column - 1] + 1 if ref_token == hyp_token\n                           else max(previous[column], current[-1]))\n        previous = current\n    return 2 * previous[-1] / (len(ref) + len(hyp))\n\n\ndef lexical_scores(reference, hypothesis):\n    """One example\'s EM, SQuAD-style Token-F1, BLEU-1/2 and ROUGE-L F1."""\n    ref, hyp = _as_text(reference), _as_text(hypothesis)\n    return {\n        "em": float(ref.casefold() == hyp.casefold()),\n        "token_f1": _token_f1(ref, hyp),\n        "bleu_1": _bleu(ref, hyp, 1),\n        "bleu_2": _bleu(ref, hyp, 2),\n        "rouge_l": _rouge_l(ref, hyp),\n    }\n\n\ndef build_bertscore_scorer(model_type="bert-base-uncased", device="cpu", batch_size=16,\n                           rescale_with_baseline=True):\n    """Create a reusable frozen semantic scorer with explicit settings."""\n    try:\n        from bert_score import BERTScorer\n    except ImportError as exc:\n        raise RuntimeError("BERTScore requires `pip install bert-score` or `pip install -r requirements.txt`") from exc\n    return BERTScorer(\n        model_type=model_type, lang="en", device=device, batch_size=batch_size,\n        rescale_with_baseline=rescale_with_baseline,\n    )\n\n\ndef score_pairs(references, hypotheses, bert_scorer):\n    """Score aligned generated answers once; return per-example metric rows."""\n    if len(references) != len(hypotheses):\n        raise ValueError("References and hypotheses must have equal lengths")\n    if bert_scorer is None:\n        raise ValueError("A BERTScore scorer is required for the six paper metrics")\n    rows = [lexical_scores(ref, hyp) for ref, hyp in zip(references, hypotheses)]\n    _add_bertscores(rows, references, hypotheses, bert_scorer, ("f1",))\n    return rows\n\n\ndef _add_bertscores(rows, references, hypotheses, bert_scorer, components):\n    """Reuse one semantic scoring call, with explicit empty-answer conventions."""\n    nonempty = [i for i, (ref, hyp) in enumerate(zip(references, hypotheses))\n                if _as_text(ref).strip() and _as_text(hyp).strip()]\n    nonempty_indices = set(nonempty)\n    for i, (ref, hyp) in enumerate(zip(references, hypotheses)):\n        if i not in nonempty_indices:\n            for component in components:\n                rows[i][f"bertscore_{component}"] = float(\n                    not _as_text(ref).strip() and not _as_text(hyp).strip())\n    if nonempty:\n        precision, recall, f1 = bert_scorer.score(\n            [_as_text(hypotheses[i]) for i in nonempty],\n            [_as_text(references[i]) for i in nonempty])\n        for component, tensor in (("precision", precision), ("recall", recall), ("f1", f1)):\n            if component not in components:\n                continue\n            values = tensor.detach().cpu().tolist() if hasattr(tensor, "detach") else list(tensor)\n            if len(values) != len(nonempty):\n                raise RuntimeError("BERTScore returned the wrong number of examples")\n            for i, value in zip(nonempty, values):\n                rows[i][f"bertscore_{component}"] = float(value)\n\n\ndef scivqa_score_pairs(references, hypotheses, bert_scorer, *, compute_bertscore=True):\n    """SciVQA Tables 1/2: ROUGE-1/L and BERTScore precision, recall and F1.\n\n    ROUGE uses rouge-score\'s lowercase alphanumeric tokenizer without stemming.\n    Scores are macro means of per-answer values when aggregated by mean_scores.\n    Answers retain source choice labels and numerical formats; no gold-dependent\n    abstention rewrite, synonym mapping, or choice reordering is applied.\n    compute_bertscore=False returns only the six ROUGE components without loading\n    a semantic model, for incremental inference journals and lexical-only runs.\n    """\n    if len(references) != len(hypotheses):\n        raise ValueError("References and hypotheses must have equal lengths")\n    if compute_bertscore and bert_scorer is None:\n        raise ValueError("A BERTScore scorer is required for the nine SciVQA metrics")\n    from rouge_score.rouge_scorer import RougeScorer\n\n    rouge = RougeScorer(["rouge1", "rougeL"], use_stemmer=False)\n    rows = []\n    for reference, hypothesis in zip(references, hypotheses):\n        scores = rouge.score(_as_text(reference), _as_text(hypothesis))\n        row = {}\n        for source, target in (("rouge1", "rouge_1"), ("rougeL", "rouge_l")):\n            score = scores[source]\n            row.update({f"{target}_f1": score.fmeasure,\n                        f"{target}_precision": score.precision,\n                        f"{target}_recall": score.recall})\n        rows.append(row)\n    if compute_bertscore:\n        _add_bertscores(rows, references, hypotheses, bert_scorer,\n                       ("precision", "recall", "f1"))\n    return rows\n\n\ndef mean_scores(rows, metrics=PAPER_METRICS):\n    if not rows:\n        raise ValueError("Cannot aggregate an empty evaluation set")\n    return {metric: fmean(row[metric] for row in rows) for metric in metrics}\n', 'utils/scivqa_prompt.py': '"""Build SciVQA inference context without reading reference answers.\n\nFigure focus and compound navigation follow sci-vqa\'s scivqa-v2 prompts.\nOnly task formatting uses qa_pair_type; answerability annotations never do.\n"""\n\nimport json\n\n\nPROMPT_VERSION = "scivqa-llm-v1"\nCONTEXT_FIELDS = ("original_question", "answer_options", "caption", "figure_type",\n                  "compound", "figs_numb", "qa_pair_type")\nPROVENANCE_FIELDS = ("figure_id", "venue", "categories", "source_dataset", "paper_id", "pdf_url")\nFIGURE_TYPE_FOCUS = {\n    "line chart": "Focus on colors of different lines and meanings, X and Y axis labels and units, scale and range of values, trends.",\n    "bar chart": "Focus on colors of different bars, heights and positions, X and Y axis labels and units.",\n    "scatter plot": "Focus on distribution of points, clusters, outliers, X and Y axis coordinates.",\n    "confusion matrix": "Focus on row and column labels, numerical values in each cell, color intensity.",\n    "pie chart": "Focus on segments and their labels, percentage or proportion values, segment colors.",\n    "box plot": "Focus on median line position, box boundaries Q1 and Q3, whisker extent, outliers.",\n    "histogram": "Focus on bin ranges, frequency heights, X and Y axis labels.",\n    "tree": "Focus on hierarchical levels, root and leaf nodes, connecting edges, labels.",\n    "graph": "Focus on nodes, vertices, edges, connectivity, labels.",\n    "architecture diagram": "Focus on component blocks, flow directions, connections, layer names.",\n    "neural networks": "Focus on layers, connections, input/output dimensions, module names.",\n}\nSYSTEM_PROMPT = (\n    "Answer the scientific question using the figure and the provided context. "\n    "Return only a concise answer, without reasoning or an \'Answer:\' prefix. "\n    "For binary questions return Yes or No. For multiple-choice questions return "\n    "only the correct option letter(s), separated by commas when several are correct. "\n    "If the answer cannot be inferred from the figure and provided context, "\n    "return \'It is not possible to answer this question based only on the provided data.\'. "\n    "Do not guess or consult external sources."\n)\n\n\ndef build_prompt(question: str, context: dict[str, str], mode: str = "full") -> str:\n    """Reconstruct exports from original_question to avoid duplicate captions/options."""\n    if mode not in {"full", "question-only"}:\n        raise ValueError(f"Unknown SciVQA context mode: {mode}")\n    text = context.get("original_question", "").strip() or question.strip()\n    raw_options = context.get("answer_options", "").strip()\n    if raw_options:\n        options = json.loads(raw_options)\n        if not isinstance(options, list):\n            raise ValueError("SciVQA answer_options must be a JSON list of label/text objects")\n        choices = {}\n        for option in options:\n            if not isinstance(option, dict) or len(option) != 1:\n                raise ValueError("Each SciVQA answer option must have exactly one label")\n            label, value = next(iter(option.items()))\n            if label in choices or label not in "ABCDEF" or len(label) != 1 or value is None:\n                raise ValueError(f"Duplicate or invalid SciVQA answer option: {label!r}")\n            choices[label] = str(value)\n        if choices and "answer options:" not in text.lower():\n            text += "\\nAnswer options: " + "; ".join(f"{key}: {value}" for key, value in choices.items())\n    kind = context.get("qa_pair_type", "").lower()\n    if "non-binary" in kind:\n        text += "\\nReturn only the letter(s) of the correct answer."\n    elif "binary" in kind:\n        text += "\\nAnswer with \'Yes\' or \'No\'."\n    # Do not branch on \'unanswerable\': that annotation reveals the target.\n    if mode == "full":\n        compound = context.get("compound", "").strip().lower() in {"true", "1", "yes"}\n        if compound and "subfigure navigation:" not in text.lower():\n            count = context.get("figs_numb", "").strip()\n            description = f"contains {count} subfigures" if count else "contains multiple subfigures"\n            text += (f"\\nSubfigure navigation: This compound figure {description}. "\n                     "Use the panel referenced in the question.")\n        figure_type = context.get("figure_type", "").strip()\n        if figure_type and "figure type:" not in text.lower():\n            focus = FIGURE_TYPE_FOCUS.get(figure_type.lower(), "Focus on colors, labels, axes, and visual elements.")\n            text += f"\\nFigure type: {figure_type} ({focus})"\n        caption = context.get("caption", "").strip()\n        if caption and "figure caption:" not in text.lower():\n            text += f"\\nFigure caption: {caption}"\n    return text\n'}
EVAL_ROOT.mkdir(parents=True, exist_ok=True)
for name, source in BUNDLED_SOURCES.items():
    destination = EVAL_ROOT / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding="utf-8")
for package in ("scripts", "utils"):
    (EVAL_ROOT / package / "__init__.py").write_text("", encoding="utf-8")
print("Restored evaluator:", EVAL_ROOT)


## Install evaluation dependencies

This keeps Kaggle's installed CUDA PyTorch/torchvision packages. All model imports
and evaluations run in fresh subprocesses, so the notebook kernel does not hold
model weights. For Gemma, accept its Hugging Face license and add `HF_TOKEN` in
Kaggle's Secrets menu, then enable `USE_KAGGLE_HF_TOKEN` above.
The evaluator first reads lightweight Hugging Face metadata, then installs the
runtime required by that architecture. OneVision 1.5 uses Transformers 4.57.1;
other presets use 5.x. Rerun this cell after changing `MODEL`. Public weights
download without a token. To authenticate, add an `HF_TOKEN` Kaggle secret and
set `USE_KAGGLE_HF_TOKEN=True`; existing environment tokens are also inherited.


In [ ]:
# Bootstrap only the Hub client before inspecting config.json (no weights yet).
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
                "huggingface-hub>=0.34,<2"], env=ENV, check=True)
profile = subprocess_json(["-c", """
import json,sys
from pathlib import Path
from scripts.evaluate_llm import inspect_model, runtime_requirements
_, config = inspect_model(sys.argv[1], sys.argv[2], Path(sys.argv[3]))
print(json.dumps({"model_type": config["model_type"], "packages": runtime_requirements(config)}))
""", str(MODEL), HF_REVISION, str(CACHE_DIR)])
print("Runtime:", profile["model_type"], profile["packages"])
packages = [
    *profile["packages"], "accelerate>=1,<2",
    "safetensors>=0.4.5", "pillow>=10", "mistral-common>=1.8.6",
]
if DATASET == "scivqa":
    packages.append("rouge-score")
if DATASET in {"plantexpert", "scivqa"} and not NO_BERTSCORE:
    packages.append("bert-score==0.3.13")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *packages], env=ENV, check=True)
subprocess.run([sys.executable, "-c", """
import torch
assert torch.cuda.device_count() == 2, "Select GPU T4 x2 in Kaggle Settings."
for i in range(2):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {p.total_memory / 2**30:.1f} GiB")
print("PyTorch:", torch.__version__)
"""], env=ENV, check=True)


## Validate data and pin the model revision

Result folders are separate for each model and dataset. Rerunning cells resumes
existing smoke/full runs with matching settings. To start a different experiment
with the same model, change `RESULTS_ROOT` or the evaluation settings and choose
an empty output folder. Changing settings requires rerunning this cell.

After a Kaggle session ends, restore the saved result folder from the Output tab
to the same `/kaggle/working/results/...` path to resume. With `HF_REVISION="main"`,
the prior run's model commit is reused. Explicit revisions are honored and checked
by the evaluator. BERTScore runs on CPU to leave both GPUs for model inference.


In [ ]:
SMOKE_OK = False
# Ask the bundled evaluator to resolve preset aliases without loading a model.
model_info = subprocess_json([
    "-c",
    "import json,sys; from scripts.evaluate_llm import parse_args; "
    "a=parse_args(['--model',sys.argv[1],'--dataset',sys.argv[2], '--split',sys.argv[3],'--scivqa-context',sys.argv[4]]); "
    "print(json.dumps({'model':a.model,'folder':a.output_dir.name}))",
    str(MODEL), DATASET, SPLIT, SCIVQA_CONTEXT,
])
SMOKE_DIR = RESULTS_ROOT / (model_info["folder"] + "_smoke")
FULL_DIR = RESULTS_ROOT / model_info["folder"]

revision = HF_REVISION
if revision == "main":
    for directory in (FULL_DIR, SMOKE_DIR):
        manifest = directory / "run_config.json"
        if manifest.is_file():
            saved = json.loads(manifest.read_text())
            if saved.get("model") == model_info["model"]:
                revision = saved.get("backend", {}).get("model_source", {}).get("revision") or revision
                break

preflight_code = """
import json, sys, torch
from pathlib import Path
from scripts.evaluate_llm import inspect_model, validate_runtime, load_examples, image_path
assert torch.cuda.device_count() == 2, "Select GPU T4 x2 in Kaggle Settings."
model, revision, cache, dtype, questions, images, split, limit, dataset, context = sys.argv[1:]
examples = load_examples(Path(questions), dataset, context)
if limit:
    examples = examples[:int(limit)]
for example in examples:
    image_path(Path(images), example, split or Path(questions).stem)
source, config = inspect_model(model, revision, Path(cache))
plan = validate_runtime(config, dtype)
print(json.dumps({"revision":source.get("revision", revision),
                  "model_type":config["model_type"], "dtype":plan[4]}))
"""
selected = subprocess_json([
    "-c", preflight_code, model_info["model"], revision,
    str(CACHE_DIR), DTYPE, str(QUESTIONS), str(IMAGES), IMAGE_SPLIT or "",
    str(max(5, FULL_LIMIT)) if FULL_LIMIT is not None else "", DATASET, SCIVQA_CONTEXT,
])
PINNED_REVISION = selected["revision"]
SETTINGS = (str(MODEL), DATASET, SPLIT, SCIVQA_CONTEXT, str(QUESTIONS), str(IMAGES), IMAGE_SPLIT, HF_REVISION,
            DTYPE, GPU_MEMORY_GIB, ATTENTION_IMPLEMENTATION, CONTEXT_SIZE, MAX_TOKENS,
            NO_BERTSCORE, FULL_LIMIT)

def evaluation_command(output_dir, limit=None):
    current = (str(MODEL), DATASET, SPLIT, SCIVQA_CONTEXT, str(QUESTIONS), str(IMAGES), IMAGE_SPLIT, HF_REVISION,
               DTYPE, GPU_MEMORY_GIB, ATTENTION_IMPLEMENTATION, CONTEXT_SIZE, MAX_TOKENS,
               NO_BERTSCORE, FULL_LIMIT)
    assert current == SETTINGS, "Settings changed: rerun the validation cell."
    command = [
        sys.executable, "-u", "-m", "scripts.evaluate_llm",
        "--model", model_info["model"], "--dataset", DATASET,
        "--split", SPLIT, "--scivqa-context", SCIVQA_CONTEXT,
        "--questions", str(QUESTIONS), "--images", str(IMAGES),
        "--output-dir", str(output_dir), "--cache-dir", str(CACHE_DIR),
        "--hf-revision", PINNED_REVISION, "--dtype", DTYPE,
        "--gpu-memory-gib", str(GPU_MEMORY_GIB),
        "--attention-implementation", ATTENTION_IMPLEMENTATION,
        "--context-size", str(CONTEXT_SIZE), "--max-tokens", str(MAX_TOKENS),
        "--bertscore-device", "cpu",
    ]
    if IMAGE_SPLIT:
        command += ["--image-split", IMAGE_SPLIT]
    if NO_BERTSCORE:
        command.append("--no-bertscore")
    if limit is not None:
        command += ["--limit", str(limit)]
    if (output_dir / "run_config.json").is_file():
        command.append("--resume")
    return command

def run_evaluation(output_dir, limit=None):
    from collections import deque
    tail = deque(maxlen=25)
    process = subprocess.Popen(evaluation_command(output_dir, limit), cwd=EVAL_ROOT, env=ENV,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, encoding="utf-8", errors="replace", bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            tail.append(line)
        code = process.wait()
    finally:
        process.stdout.close()
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
    if code:
        raise RuntimeError(f"Evaluation failed (exit {code}); saved results: {output_dir}\n"
                           + "".join(tail))
    report = json.loads((output_dir / "report.json").read_text())
    placement = set(report["config"]["backend"]["device_map"].values())
    assert placement <= {"0", "1", "cuda:0", "cuda:1"}, "Unexpected CPU/disk placement."
    assert ({"0", "cuda:0"} & placement and {"1", "cuda:1"} & placement), "Model was not split across both GPUs."
    assert report["complete"], "Run is incomplete; check its report and resume."
    return report

print("Architecture:", selected["model_type"], "| dtype:", selected["dtype"])
print("Pinned revision:", PINNED_REVISION)
print("Full results:", FULL_DIR)


## Smoke test, then full evaluation

The subprocess exits after each run and releases its GPU memory. Full evaluation
begins only when the smoke run succeeds. To resume an existing full run without
repeating the smoke run, set `RUN_SMOKE=False`. If inference fails, the exception
shows the final output lines; completed predictions remain saved for resuming.


In [ ]:
SMOKE_OK = not RUN_SMOKE
if RUN_SMOKE:
    smoke_report = run_evaluation(SMOKE_DIR, limit=5)
    SMOKE_OK = True
    print("Smoke evaluation completed; ready for the full run.")


In [ ]:
assert SMOKE_OK, "Run the smoke cell successfully before full evaluation."
full_report = run_evaluation(FULL_DIR, limit=FULL_LIMIT)


In [ ]:
from IPython.display import display, JSON

summary = {key: full_report[key] for key in (
    "examples_selected", "examples_succeeded", "errors", "complete",
    "truncated_responses", "mean_latency_seconds",
)}
if DATASET in {"plantexpert", "scivqa"}:
    summary["metrics"] = full_report["generated_metrics"]
else:
    summary["accuracy"] = full_report["accuracy"]
    summary["normalized_accuracy"] = full_report["normalized_accuracy"]
summary["gpu_placement"] = full_report["config"]["backend"]["device_map"]
display(JSON(summary))
print("Download these files from Kaggle's Output tab after saving a version:")
for name in ("report.json", "predictions.jsonl", "run_config.json"):
    print(FULL_DIR / name)
